# Exercises Pack — MCP Client with an LLM

## Complete beginner solution using STDIO

This notebook connects three separate ideas:

1. an MCP server advertises typed tools;
2. a planner chooses which tool should be called;
3. the MCP client validates and executes that call.

The default planner is a local deterministic stub, so the full exercise runs
without tokens or paid APIs. An optional GitHub Models branch is included
for students who provide a `GITHUB_TOKEN`.

## What you will build

```text
User prompt: "Add 2 to 20"
                ↓
       Stub or real LLM planner
                ↓
   {"name": "add", "args": {"a": 2, "b": 20}}
                ↓
         MCP ClientSession
                ↓ STDIO
          MCP DemoServer
                ↓
               22
```

The planner never executes Python directly. It proposes a tool name and
arguments. The client remains responsible for validation and execution.

## Learning objectives

You will learn how to:

- explain why STDIO is simple for local MCP development;
- initialize a `ClientSession`;
- discover tools, resources, and resource templates;
- inspect MCP `inputSchema` values;
- convert an MCP tool to an LLM function specification;
- build a no-token stub planner;
- optionally call GitHub Models;
- validate LLM-generated tool calls;
- execute those calls through MCP;
- parse and display tool results.

# Setup — Install the stable MCP SDK

In [ ]:
# The stable MCP Python SDK is currently the v1.x line.
# `<2` avoids an accidental upgrade to an incompatible major release.
#
# `requests` is needed only for the optional GitHub Models branch.

%pip install -qU \
    "mcp[cli]>=1.28,<2" \
    "requests>=2.31,<3"

In [ ]:
# Imports used throughout the exercises.

import asyncio
import copy
import importlib.metadata as metadata
import json
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path
from typing import Any

import requests
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client

print("Python:", sys.version.split()[0])
print("MCP SDK:", metadata.version("mcp"))

assert sys.version_info >= (3, 10)

In [ ]:
# Configuration.
#
# Stub mode is deliberately the default.
USE_REAL_LLM = False

# Real mode requires a GitHub token with models:read permission.
GITHUB_MODEL = os.getenv(
    "GITHUB_MODEL",
    "openai/gpt-4.1",
)

# Resolve server.py from the current notebook directory.
SERVER_PATH = Path("server.py").resolve()

print("Planner mode:", "real LLM" if USE_REAL_LLM else "stub")
print("Server path:", SERVER_PATH)

# Create the MCP server

In [ ]:
%%writefile server.py
"""Small MCP calculator server for the LLM-planning exercises.

The server itself contains no LLM. It exposes typed MCP capabilities that a
separate client can discover and present to either:
- a deterministic local stub planner; or
- an optional remote LLM planner.

STDIO note:
Do not print normal logs to stdout in a STDIO MCP server. Stdout is reserved
for MCP JSON-RPC messages.
"""

from mcp.server.fastmcp import FastMCP


# FastMCP uses Python annotations and docstrings to build JSON schemas.
mcp = FastMCP("DemoServer")


@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two integers.

    Args:
        a: First integer.
        b: Second integer.

    Returns:
        The sum a + b.
    """
    return a + b


@mcp.tool()
def multiply(a: int, b: int) -> int:
    """Multiply two integers.

    This is the optional exercise implemented as a bonus.

    Args:
        a: First integer.
        b: Second integer.

    Returns:
        The product a * b.
    """
    return a * b


@mcp.resource("greeting://{name}")
def greet(name: str) -> str:
    """Return a personalized read-only greeting resource."""
    return f"Hello, {name}!"


def main() -> None:
    """Start the local MCP server using STDIO."""
    mcp.run(transport="stdio")


if __name__ == "__main__":
    main()

## Server notes

- `add` is the required tool.
- `multiply` completes the optional extension.
- `greeting://{name}` is a read-only resource template.
- `mcp.run(transport="stdio")` starts the local protocol loop.
- The server contains no LLM.

In [ ]:
# Validate syntax before attempting an MCP connection.

import py_compile

py_compile.compile(
    "server.py",
    doraise=True,
)

print("server.py syntax: OK")

# Exercise 1 — Why STDIO is simpler than HTTP locally

STDIO is simpler because the client starts the server as a child process and
communicates through standard input and output. There is no port to choose,
no web server to configure, no routing layer, and usually no local
authentication setup. The operating system already manages the process and
pipes, so the server can start and stop with the client.

HTTP is more appropriate when the server must be remote or shared by many
clients, but it introduces networking, binding addresses, authentication,
TLS, deployment, and firewall concerns.

# Exercise 2 — Connect and initialize

In [ ]:
def find_mcp_command() -> str:
    """Return the MCP CLI path or raise a clear setup error."""
    command = shutil.which("mcp")

    if command is None:
        raise RuntimeError(
            "The mcp command is unavailable. Re-run the install cell."
        )

    return command


def make_server_params() -> StdioServerParameters:
    """Describe how the client should start server.py."""
    return StdioServerParameters(
        command=find_mcp_command(),
        args=["run", str(SERVER_PATH)],
        env=os.environ.copy(),
    )

In [ ]:
async def ex2_connect() -> None:
    """Start the server and complete the MCP initialization handshake."""
    params = make_server_params()

    # stdio_client launches the child process and exposes protocol streams.
    async with stdio_client(params) as (
        read_stream,
        write_stream,
    ):
        # ClientSession implements the MCP lifecycle.
        async with ClientSession(
            read_stream,
            write_stream,
        ) as session:
            initialization = await session.initialize()

            print(
                "Connected and initialized:",
                initialization.serverInfo.name,
            )


await ex2_connect()
print("Exercise 2: OK")

# Exercise 3 — Discover resources and tools

In [ ]:
def get_input_schema(tool: Any) -> dict[str, Any]:
    """Read an MCP tool input schema across SDK naming conventions."""
    schema = getattr(tool, "inputSchema", None)

    if schema is None:
        schema = getattr(tool, "input_schema", None)

    return copy.deepcopy(schema or {
        "type": "object",
        "properties": {},
    })


def resource_template_uri(template: Any) -> str:
    """Extract the URI template from the MCP SDK object."""
    value = getattr(template, "uriTemplate", None)

    if value is None:
        value = getattr(template, "uri_template", None)

    return str(value)

In [ ]:
async def ex3_list() -> list[Any]:
    """List capabilities and return the discovered MCP Tool objects."""
    params = make_server_params()

    async with stdio_client(params) as (
        read_stream,
        write_stream,
    ):
        async with ClientSession(
            read_stream,
            write_stream,
        ) as session:
            await session.initialize()

            resources_result = await session.list_resources()
            templates_result = (
                await session.list_resource_templates()
            )
            tools_result = await session.list_tools()

            print("STATIC RESOURCES:", [
                str(resource.uri)
                for resource in resources_result.resources
            ])

            print("RESOURCE TEMPLATES:", [
                resource_template_uri(template)
                for template
                in templates_result.resourceTemplates
            ])

            print("TOOLS:")

            for tool in tools_result.tools:
                properties = get_input_schema(tool).get(
                    "properties",
                    {},
                )

                print(
                    f"- {tool.name}: {properties}"
                )

            return tools_result.tools


discovered_tools = await ex3_list()

## Why the static resource list is empty

`greeting://{name}` is dynamic. It is therefore returned by
`list_resource_templates()`, not `list_resources()`.

Examples instantiated from that template include:

- `greeting://Fahim`
- `greeting://student`
- `greeting://hello`

# Exercise 4 — Convert MCP tools to LLM function specs

## What is being converted?

MCP discovery returns a tool with:

- a name;
- a description;
- an `inputSchema`.

Most chat-completion APIs expect an outer `type: function` wrapper. The
parameter schema itself is already JSON Schema, so it can be preserved.

In [ ]:
def convert_to_llm_tool(
    tool: Any,
) -> dict[str, Any]:
    """Convert one MCP Tool into an LLM function-tool specification."""
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": (
                tool.description
                or "MCP tool"
            ),
            # Preserve the entire MCP schema, including required fields.
            "parameters": get_input_schema(tool),
        },
    }


llm_functions = [
    convert_to_llm_tool(tool)
    for tool in discovered_tools
]

print(
    json.dumps(
        llm_functions,
        indent=2,
    )
)

## Important boundary

`convert_to_llm_tool` does not run a tool. It only tells the planner:

- which functions exist;
- what each function does;
- which arguments are valid.

The `ClientSession` performs execution later.

# Exercise 5 — Plan and execute

## Deterministic stub planner

In [ ]:
def stub_plan(
    prompt: str,
    functions: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """Generate a predictable tool call without a remote LLM."""
    available_tools = {
        item["function"]["name"]
        for item in functions
    }

    numbers = [
        int(value)
        for value in re.findall(r"-?\d+", prompt)
    ]

    if len(numbers) < 2:
        raise ValueError(
            "Please include at least two integers in the prompt."
        )

    lower_prompt = prompt.lower()

    multiplication_words = {
        "multiply",
        "multiplied",
        "times",
        "product",
    }

    if any(
        word in lower_prompt
        for word in multiplication_words
    ):
        tool_name = "multiply"
    else:
        tool_name = "add"

    if tool_name not in available_tools:
        raise ValueError(
            f"{tool_name!r} is not advertised by the MCP server."
        )

    return [{
        "name": tool_name,
        "args": {
            "a": numbers[0],
            "b": numbers[1],
        },
    }]

## Optional GitHub Models planner

In [ ]:
GITHUB_MODELS_URL = (
    "https://models.github.ai/inference/chat/completions"
)
GITHUB_API_VERSION = "2026-03-10"


def github_models_plan(
    prompt: str,
    functions: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """Ask GitHub Models for function calls."""
    token = os.getenv("GITHUB_TOKEN")

    if not token:
        raise RuntimeError(
            "Set GITHUB_TOKEN or keep USE_REAL_LLM=False."
        )

    response = requests.post(
        GITHUB_MODELS_URL,
        headers={
            "Accept": "application/vnd.github+json",
            "Authorization": f"Bearer {token}",
            "X-GitHub-Api-Version": GITHUB_API_VERSION,
            "Content-Type": "application/json",
        },
        json={
            "model": GITHUB_MODEL,
            "messages": [
                {
                    "role": "system",
                    "content": (
                        "Select the correct provided function. "
                        "Return tool calls only."
                    ),
                },
                {
                    "role": "user",
                    "content": prompt,
                },
            ],
            "tools": functions,
            "tool_choice": "required",
            "temperature": 0,
            "max_tokens": 300,
        },
        timeout=60,
    )
    response.raise_for_status()

    message = response.json()["choices"][0]["message"]
    raw_calls = message.get("tool_calls") or []

    calls = []

    for raw_call in raw_calls:
        function = raw_call["function"]
        raw_arguments = function.get("arguments", {})

        arguments = (
            json.loads(raw_arguments)
            if isinstance(raw_arguments, str)
            else raw_arguments
        )

        calls.append({
            "name": function["name"],
            "args": arguments,
        })

    if not calls:
        raise RuntimeError(
            "The LLM returned no tool calls."
        )

    return calls

In [ ]:
def call_llm(
    prompt: str,
    functions: list[dict[str, Any]],
    use_real: bool = False,
) -> list[dict[str, Any]]:
    """Route planning to the selected implementation."""
    if use_real:
        return github_models_plan(
            prompt,
            functions,
        )

    return stub_plan(
        prompt,
        functions,
    )

## Safe result parsing and call validation

In [ ]:
def extract_tool_value(result: Any) -> Any:
    """Extract a typed or text value from CallToolResult."""
    structured = getattr(result, "structuredContent", None)

    if structured is None:
        structured = getattr(result, "structured_content", None)

    if isinstance(structured, dict):
        if set(structured) == {"result"}:
            return structured["result"]

        return structured

    for block in getattr(result, "content", []):
        if isinstance(block, types.TextContent):
            return block.text

        text = getattr(block, "text", None)
        if text is not None:
            return text

    return str(result)


def validate_planned_call(
    call: dict[str, Any],
    known_tool_names: set[str],
) -> None:
    """Reject unknown tool names or malformed arguments."""
    if not isinstance(call, dict):
        raise TypeError(
            "Each planned call must be a dictionary."
        )

    name = call.get("name")
    arguments = call.get("args")

    if name not in known_tool_names:
        raise ValueError(
            f"Unknown planned tool: {name!r}"
        )

    if not isinstance(arguments, dict):
        raise TypeError(
            f"Arguments for {name!r} must be a dictionary."
        )

In [ ]:
async def ex5_run(
    prompt: str = "Add 2 to 20.",
) -> list[dict[str, Any]]:
    """Discover, convert, plan, validate, execute, and print."""
    params = make_server_params()

    async with stdio_client(params) as (
        read_stream,
        write_stream,
    ):
        async with ClientSession(
            read_stream,
            write_stream,
        ) as session:
            await session.initialize()

            tools_result = await session.list_tools()
            tools = tools_result.tools

            functions = [
                convert_to_llm_tool(tool)
                for tool in tools
            ]

            calls = call_llm(
                prompt,
                functions,
                use_real=USE_REAL_LLM,
            )

            known_tool_names = {
                tool.name
                for tool in tools
            }

            print("Prompt:", prompt)
            print(
                "Planner:",
                "GitHub Models"
                if USE_REAL_LLM
                else "deterministic stub",
            )
            print("tool_calls:", calls)

            execution_records = []

            for call in calls:
                validate_planned_call(
                    call,
                    known_tool_names,
                )

                result = await session.call_tool(
                    call["name"],
                    arguments=call["args"],
                )
                value = extract_tool_value(result)

                record = {
                    "tool": call["name"],
                    "arguments": call["args"],
                    "result": value,
                }
                execution_records.append(record)

                print("result:", record)

            return execution_records

In [ ]:
# Required demonstration.
addition_records = await ex5_run(
    "Add 2 to 20."
)

assert str(addition_records[0]["result"]) == "22"

# Optional exercise — Multiply

In [ ]:
# The optional multiply tool is already registered on the server.
# The same discovery/conversion/planning/execution pipeline handles it.

multiplication_records = await ex5_run(
    "Multiply 6 times 7."
)

assert str(multiplication_records[0]["result"]) == "42"

# How the conversion and execution fit together

```text
MCP server
  add(a: int, b: int)
          ↓ tools/list
MCP Tool object
  name + description + inputSchema
          ↓ convert_to_llm_tool
LLM function specification
          ↓ stub or real planner
Proposed tool call
  {"name": "add", "args": {"a": 2, "b": 20}}
          ↓ validation
session.call_tool(...)
          ↓
MCP result: 22
```

MCP standardizes discovery and execution. The LLM remains replaceable.

# Create a standalone client submission file

In [ ]:
%%writefile client.py
"""MCP client with a stub or optional GitHub Models planner.

Flow:
1. start server.py over STDIO;
2. initialize an MCP session;
3. discover resource templates and tools;
4. convert MCP tool schemas into LLM function specifications;
5. ask a stub or real LLM to propose tool calls;
6. validate and execute those calls through MCP;
7. print the results.

Stub mode is the default and needs no token.
"""

import asyncio
import copy
import json
import os
import re
import shutil
from pathlib import Path
from typing import Any

import requests
from mcp import ClientSession, StdioServerParameters, types
from mcp.client.stdio import stdio_client


SERVER_PATH = Path(__file__).resolve().with_name("server.py")

# Keep the exercise free and deterministic by default.
USE_REAL_LLM = os.getenv("USE_REAL_LLM", "false").lower() == "true"

# Current GitHub Models REST endpoint and an example model identifier.
GITHUB_MODELS_URL = (
    "https://models.github.ai/inference/chat/completions"
)
GITHUB_API_VERSION = "2026-03-10"
GITHUB_MODEL = os.getenv(
    "GITHUB_MODEL",
    "openai/gpt-4.1",
)


def find_mcp_command() -> str:
    """Locate the MCP CLI in the active Python environment."""
    command = shutil.which("mcp")

    if command is None:
        raise RuntimeError(
            "The 'mcp' command was not found. Activate the environment "
            "and install 'mcp[cli]>=1.28,<2'."
        )

    return command


def make_server_params() -> StdioServerParameters:
    """Describe the child process used as the MCP server."""
    return StdioServerParameters(
        command=find_mcp_command(),
        args=["run", str(SERVER_PATH)],
        env=os.environ.copy(),
    )


def get_input_schema(tool: Any) -> dict[str, Any]:
    """Read an MCP tool input schema across compatible SDK field names."""
    schema = getattr(tool, "inputSchema", None)

    if schema is None:
        schema = getattr(tool, "input_schema", None)

    return copy.deepcopy(schema or {
        "type": "object",
        "properties": {},
    })


def convert_to_llm_tool(tool: Any) -> dict[str, Any]:
    """Convert one MCP Tool object into an LLM function-tool schema.

    MCP already describes tool arguments with JSON Schema. The conversion
    mainly changes the outer envelope expected by chat-completion APIs.

    MCP:
        Tool(name, description, inputSchema)

    LLM:
        {
          "type": "function",
          "function": {
            "name": ...,
            "description": ...,
            "parameters": ...
          }
        }
    """
    return {
        "type": "function",
        "function": {
            "name": tool.name,
            "description": (
                tool.description
                or "MCP tool"
            ),
            # Preserve the complete schema, not only its properties.
            "parameters": get_input_schema(tool),
        },
    }


def stub_plan(
    prompt: str,
    functions: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """Create deterministic tool calls without using an LLM.

    This beginner stub recognizes addition and multiplication language,
    extracts the first two integers, and verifies that the selected tool is
    actually present in the function list discovered from MCP.
    """
    available_tools = {
        item["function"]["name"]
        for item in functions
    }

    numbers = [
        int(value)
        for value in re.findall(r"-?\d+", prompt)
    ]

    if len(numbers) < 2:
        raise ValueError(
            "The stub planner needs at least two integers in the prompt."
        )

    normalized_prompt = prompt.lower()

    multiplication_words = {
        "multiply",
        "multiplied",
        "times",
        "product",
    }

    if any(
        word in normalized_prompt
        for word in multiplication_words
    ):
        selected_tool = "multiply"
    else:
        selected_tool = "add"

    if selected_tool not in available_tools:
        raise ValueError(
            f"The planner selected {selected_tool!r}, "
            "but the MCP server did not advertise that tool."
        )

    return [
        {
            "name": selected_tool,
            "args": {
                "a": numbers[0],
                "b": numbers[1],
            },
        }
    ]


def github_models_plan(
    prompt: str,
    functions: list[dict[str, Any]],
) -> list[dict[str, Any]]:
    """Ask GitHub Models to propose function calls.

    Requirements:
    - set GITHUB_TOKEN;
    - the token needs `models:read`;
    - set USE_REAL_LLM=true to opt in.

    The MCP server is still invoked by this client. The LLM only proposes
    names and JSON arguments.
    """
    token = os.getenv("GITHUB_TOKEN")

    if not token:
        raise RuntimeError(
            "Set GITHUB_TOKEN or keep USE_REAL_LLM=false."
        )

    response = requests.post(
        GITHUB_MODELS_URL,
        headers={
            "Accept": "application/vnd.github+json",
            "Authorization": f"Bearer {token}",
            "X-GitHub-Api-Version": GITHUB_API_VERSION,
            "Content-Type": "application/json",
        },
        json={
            "model": GITHUB_MODEL,
            "messages": [
                {
                    "role": "system",
                    "content": (
                        "Choose the correct provided function for the "
                        "user request. Return tool calls only."
                    ),
                },
                {
                    "role": "user",
                    "content": prompt,
                },
            ],
            "tools": functions,
            "tool_choice": "required",
            "temperature": 0,
            "max_tokens": 300,
        },
        timeout=60,
    )
    response.raise_for_status()

    payload = response.json()
    message = payload["choices"][0]["message"]
    raw_tool_calls = message.get("tool_calls") or []

    calls: list[dict[str, Any]] = []

    for tool_call in raw_tool_calls:
        function = tool_call["function"]
        raw_arguments = function.get("arguments", {})

        if isinstance(raw_arguments, str):
            arguments = json.loads(raw_arguments)
        else:
            arguments = raw_arguments

        calls.append({
            "name": function["name"],
            "args": arguments,
        })

    if not calls:
        raise RuntimeError(
            "The real LLM returned no tool calls."
        )

    return calls


def call_llm(
    prompt: str,
    functions: list[dict[str, Any]],
    use_real: bool = False,
) -> list[dict[str, Any]]:
    """Select the real or deterministic planning implementation."""
    if use_real:
        return github_models_plan(
            prompt,
            functions,
        )

    return stub_plan(
        prompt,
        functions,
    )


def resource_template_uri(template: Any) -> str:
    """Read a resource-template URI across SDK naming styles."""
    value = getattr(template, "uriTemplate", None)

    if value is None:
        value = getattr(template, "uri_template", None)

    return str(value)


def extract_tool_value(result: Any) -> Any:
    """Extract structured or text data from a CallToolResult."""
    structured = getattr(result, "structuredContent", None)

    if structured is None:
        structured = getattr(result, "structured_content", None)

    if isinstance(structured, dict):
        if set(structured) == {"result"}:
            return structured["result"]

        return structured

    for block in getattr(result, "content", []):
        if isinstance(block, types.TextContent):
            return block.text

        text = getattr(block, "text", None)
        if text is not None:
            return text

    return str(result)


def validate_planned_call(
    call: dict[str, Any],
    known_tools: dict[str, Any],
) -> None:
    """Reject unknown names or malformed planner output before execution."""
    if not isinstance(call, dict):
        raise TypeError("Each planned call must be a dictionary.")

    name = call.get("name")
    arguments = call.get("args")

    if name not in known_tools:
        raise ValueError(
            f"Planner proposed unknown tool: {name!r}"
        )

    if not isinstance(arguments, dict):
        raise TypeError(
            f"Arguments for {name!r} must be a dictionary."
        )


async def run(
    prompt: str = "Add 2 to 20.",
) -> None:
    """Discover tools, plan calls, execute them, and print results."""
    params = make_server_params()

    async with stdio_client(params) as (
        read_stream,
        write_stream,
    ):
        async with ClientSession(
            read_stream,
            write_stream,
        ) as session:
            initialization = await session.initialize()

            print(
                "Connected server:",
                initialization.serverInfo.name,
            )

            resources_result = await session.list_resources()
            static_resources = [
                str(resource.uri)
                for resource in resources_result.resources
            ]

            templates_result = (
                await session.list_resource_templates()
            )
            resource_templates = [
                resource_template_uri(template)
                for template
                in templates_result.resourceTemplates
            ]

            tools_result = await session.list_tools()
            known_tools = {
                tool.name: tool
                for tool in tools_result.tools
            }

            print("Static resources:", static_resources)
            print("Resource templates:", resource_templates)
            print("Tools and input properties:")

            for tool in tools_result.tools:
                properties = get_input_schema(tool).get(
                    "properties",
                    {},
                )
                print(
                    f"- {tool.name}: {properties}"
                )

            # Convert every MCP tool to the function format given to the
            # stub or real LLM.
            functions = [
                convert_to_llm_tool(tool)
                for tool in tools_result.tools
            ]

            print("LLM function names:", [
                function["function"]["name"]
                for function in functions
            ])

            calls = call_llm(
                prompt,
                functions,
                use_real=USE_REAL_LLM,
            )

            planner_name = (
                "GitHub Models"
                if USE_REAL_LLM
                else "deterministic stub"
            )

            print("Planner:", planner_name)
            print("Prompt:", prompt)
            print("tool_calls:", calls)

            for call in calls:
                validate_planned_call(
                    call,
                    known_tools,
                )

                result = await session.call_tool(
                    call["name"],
                    arguments=call["args"],
                )
                value = extract_tool_value(result)

                print(
                    f"result {call['name']}"
                    f"({call['args']}) -> {value}"
                )


def main() -> None:
    """Run the asynchronous MCP client from a normal terminal."""
    prompt = os.getenv(
        "DEMO_PROMPT",
        "Add 2 to 20.",
    )
    asyncio.run(run(prompt))


if __name__ == "__main__":
    main()

In [ ]:
# Validate both files before the end-to-end terminal test.

for filename in ["server.py", "client.py"]:
    py_compile.compile(
        filename,
        doraise=True,
    )
    print(f"{filename} syntax: OK")

# Run the standalone client and capture output

In [ ]:
completed = subprocess.run(
    [sys.executable, "client.py"],
    capture_output=True,
    text=True,
    timeout=30,
    check=False,
    env={
        **os.environ,
        # Force the free stub branch for the submitted capture.
        "USE_REAL_LLM": "false",
        "DEMO_PROMPT": "Add 2 to 20.",
    },
)

print("CLIENT STDOUT")
print(completed.stdout)

if completed.stderr.strip():
    print("SERVER/CLIENT STDERR")
    print(completed.stderr)

print("Return code:", completed.returncode)

assert completed.returncode == 0

In [ ]:
terminal_capture = (
    "$ python client.py\n"
    + completed.stdout
)

Path("mcp_llm_terminal_capture.txt").write_text(
    terminal_capture,
    encoding="utf-8",
)

print(terminal_capture)
print("Saved: mcp_llm_terminal_capture.txt")

## Expected capture

```text
Connected server: DemoServer
Static resources: []
Resource templates: ['greeting://{name}']
Tools and input properties:
- add: {'a': ..., 'b': ...}
- multiply: {'a': ..., 'b': ...}
LLM function names: ['add', 'multiply']
Planner: deterministic stub
Prompt: Add 2 to 20.
tool_calls: [{'name': 'add', 'args': {'a': 2, 'b': 20}}]
result add({'a': 2, 'b': 20}) -> 22
```

# Observations

- STDIO removes local networking setup.
- MCP discovery supplies JSON schemas dynamically.
- The schema conversion is small because MCP already uses JSON Schema.
- Stub mode is deterministic, free, and suitable for grading.
- A real LLM may propose invalid or unexpected calls, so validation remains
  necessary.
- Tool execution belongs to the client, not the LLM.
- Adding `multiply` requires no planner/executor rewrite because the tool
  list is discovered dynamically.

# Troubleshooting

## `mcp: command not found`

Re-run the install cell or activate the correct environment.

## Connection closes immediately

Compile the files:

```bash
python -m py_compile server.py client.py
```

A STDIO server must not print normal logs to stdout.

## No tool calls in real mode

Confirm:

- `USE_REAL_LLM = True`;
- `GITHUB_TOKEN` exists;
- the token has `models:read`;
- the selected GitHub model supports tools.

## Wrong arguments

Print the discovered `inputSchema` and validate the planner output before
calling `session.call_tool`.

# Deliverables checklist

- [x] Exercise 1 theory answer
- [x] STDIO server startup
- [x] ClientSession initialization
- [x] Resource discovery
- [x] Resource-template discovery
- [x] Tool discovery
- [x] Input-schema property display
- [x] `convert_to_llm_tool`
- [x] No-token stub planner
- [x] Optional GitHub Models planner
- [x] Planned-call validation
- [x] MCP tool execution
- [x] `Add 2 to 20` output
- [x] Optional multiply tool
- [x] Standalone `server.py`
- [x] Standalone `client.py`
- [x] Terminal capture
- [x] Thorough comments and documentation

# References

- Stable MCP Python SDK:
  https://github.com/modelcontextprotocol/python-sdk/tree/v1.x
- MCP architecture:
  https://modelcontextprotocol.io/docs/learn/architecture
- GitHub Models prototyping:
  https://docs.github.com/en/github-models/use-github-models/prototyping-with-ai-models
- GitHub Models inference REST API:
  https://docs.github.com/en/rest/models/inference